<a href="https://colab.research.google.com/github/andreendrizzi-coder/App_Produccion/blob/main/Reporte_Produccion.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 0. Instalamos las herramientas necesarias
!pip install pymssql fpdf pytz matplotlib

import pandas as pd
from fpdf import FPDF
from datetime import datetime
import pytz
from google.colab import userdata
import pymssql
import warnings
import matplotlib.pyplot as plt
import numpy as np

# Ocultar advertencias amarillas de Pandas
warnings.filterwarnings('ignore', category=UserWarning)

# ==============================================================
# 📅 1. DEFINE EL PERÍODO DEL REPORTE AQUÍ (Formato AAAA-MM-DD)
# ==============================================================
FECHA_INICIO = '2026-10-01'
FECHA_FIN    = '2026-10-31'
# ==============================================================

print(f"Conectando a Wiidem para extraer datos del {FECHA_INICIO} al {FECHA_FIN}...")

try:
    # 2. CONEXIÓN SEGURA A SQL
    conexion = pymssql.connect(
        server='200.49.98.250',
        user=userdata.get('WIIDEM_USER'),
        password=userdata.get('WIIDEM_PASS'),
        database=userdata.get('WIIDEM_DB'),
        login_timeout=15
    )

    # 3. CONSULTA 1: OEE GENERAL (Promediado por el rango de fechas)
    query_oee = f"""
    SELECT
        Cell AS MAQUINA,
        AVG(Oee) AS OEE,
        AVG(Availability) AS DISPONIBILIDAD,
        AVG(Performance) AS PERFORMANCE,
        AVG(Quality) AS CALIDAD
    FROM PROD_D_03
    WHERE Date >= '{FECHA_INICIO}' AND Date <= '{FECHA_FIN}'
      AND Cell LIKE 'CEL%'
    GROUP BY Cell
    ORDER BY Cell
    """
    df_oee = pd.read_sql(query_oee, conexion)

    # 4. CONSULTA 2: METRICAS DE TURNO (Productivo y Parada desde PROD_D_02)
    query_prod_turnos = f"""
    SELECT
        Cell AS MAQUINA,
        Turn AS TURNO,
        SUM(ProductiveTime) AS PRODUCTIVO,
        SUM(DownTime) AS PARADA
    FROM PROD_D_02
    WHERE Date >= '{FECHA_INICIO}' AND Date <= '{FECHA_FIN}'
      AND Cell LIKE 'CEL%'
    GROUP BY Cell, Turn
    ORDER BY Turn, Cell
    """
    df_prod_turnos = pd.read_sql(query_prod_turnos, conexion)

    # 5. CONSULTA 3: HORARIOS DE INICIO Y CIERRE (Desde EVENT_01)
    query_horarios = f"""
    SELECT
        Cell AS MAQUINA,
        Turn AS TURNO,
        MIN(Started) AS HORA_INICIO,
        MAX(Finish) AS HORA_CIERRE
    FROM EVENT_01
    WHERE Started >= '{FECHA_INICIO}' AND Started <= '{FECHA_FIN} 23:59:59'
      AND Cell LIKE 'CEL%'
    GROUP BY Cell, Turn
    """
    df_horarios = pd.read_sql(query_horarios, conexion)
    conexion.close()

    print("✅ Datos extraídos exitosamente. Procesando gráficos y tablas...")

    # ---------------------------------------------------------
    # FASE DE PROCESAMIENTO DE DATOS
    # ---------------------------------------------------------
    def formatear_nombre_maq(nombre):
        return str(nombre).replace('CELDA ', 'Celda 0').replace('CEL', 'Celda ') + ' Fumis'

    # A) Procesar OEE
    df_oee['MAQUINA'] = df_oee['MAQUINA'].apply(formatear_nombre_maq)
    cols_kpi = ['OEE', 'DISPONIBILIDAD', 'PERFORMANCE', 'CALIDAD']
    for col in cols_kpi:
        df_oee[col] = pd.to_numeric(df_oee[col], errors='coerce').fillna(0.0)

    total_grupo = df_oee[cols_kpi].mean() if not df_oee.empty else None

    # B) Procesar Tabla de Turnos (Sección 3) - AGRUPADO POR TURNO PRIMERO
    df_turnos = pd.merge(df_prod_turnos, df_horarios, on=['MAQUINA', 'TURNO'], how='left')
    df_turnos['MAQUINA'] = df_turnos['MAQUINA'].apply(formatear_nombre_maq)

    # ORDENADO PRIMERO POR TURNO Y LUEGO POR MÁQUINA
    df_turnos = df_turnos.sort_values(by=['TURNO', 'MAQUINA'])

    # Convertir a datetime para operaciones
    df_turnos['HORA_INICIO_DT'] = pd.to_datetime(df_turnos['HORA_INICIO'])
    df_turnos['HORA_CIERRE_DT'] = pd.to_datetime(df_turnos['HORA_CIERRE'])

    df_turnos['TXT_INICIO'] = df_turnos['HORA_INICIO_DT'].dt.strftime('%H:%M').fillna('-')
    df_turnos['TXT_CIERRE'] = df_turnos['HORA_CIERRE_DT'].dt.strftime('%H:%M').fillna('-')

    def min_a_hhmm_hs(minutos_num):
        if pd.isna(minutos_num) or minutos_num <= 0:
            return "00:00 hs"
        hrs = int(minutos_num // 60)
        mins = int(minutos_num % 60)
        return f"{hrs:02d}:{mins:02d} hs"

    # Apertura Neta = Productivo + Paradas
    df_turnos['MIN_APERTURA_NETA'] = df_turnos['PRODUCTIVO'].fillna(0) + df_turnos['PARADA'].fillna(0)
    df_turnos['TXT_APERTURA_NETA'] = df_turnos['MIN_APERTURA_NETA'].apply(min_a_hhmm_hs)

    # Tiempo No Registrado = (Cierre - Inicio en mins) - Apertura Neta
    def calc_no_registrado(row):
        if pd.isna(row['HORA_INICIO_DT']) or pd.isna(row['HORA_CIERRE_DT']):
            return "-"
        duracion_total_min = (row['HORA_CIERRE_DT'] - row['HORA_INICIO_DT']).total_seconds() / 60.0
        no_reg = duracion_total_min - row['MIN_APERTURA_NETA']
        if no_reg > 0.5:
            return min_a_hhmm_hs(no_reg)
        elif no_reg >= 0:
            return "00:00 hs"
        else:
            return "-"

    df_turnos['TXT_NO_REGISTRADO'] = df_turnos.apply(calc_no_registrado, axis=1)

    # ---------------------------------------------------------
    # FASE DE GENERACIÓN DEL GRÁFICO (MATPLOTLIB)
    # ---------------------------------------------------------
    if not df_oee.empty:
        maquinas = df_oee['MAQUINA'].tolist()
        fig, ax = plt.subplots(figsize=(10, 4.5))

        x = np.arange(len(cols_kpi))
        width = 0.8 / len(maquinas)
        colores = plt.cm.tab20(np.linspace(0, 1, len(maquinas)))

        for i, maquina in enumerate(maquinas):
            valores = df_oee[df_oee['MAQUINA'] == maquina][cols_kpi].values[0]
            offset = (i - len(maquinas)/2) * width + width/2
            rects = ax.bar(x + offset, valores, width, label=maquina, color=colores[i])
            ax.bar_label(rects, fmt='%.1f', padding=2, rotation=90, fontsize=8, color='black')

        ax.set_ylabel('Porcentaje (%)')
        ax.set_xticks(x)
        ax.set_xticklabels(cols_kpi)

        max_valor_real = df_oee[cols_kpi].max().max()
        ax.set_ylim(0, max_valor_real * 1.4)
        ax.legend(bbox_to_anchor=(0.5, 1.20), loc='upper center', ncol=4, frameon=False, fontsize=8.5)

        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(axis='y', linestyle='--', alpha=0.5)

        plt.tight_layout()
        plt.savefig('grafico_kpis.png', dpi=300, bbox_inches='tight')
        plt.close()

    # ---------------------------------------------------------
    # FASE DE GENERACIÓN DEL PDF
    # ---------------------------------------------------------
    COLOR_TITULO = (211, 84, 0)
    COLOR_GRIS = (100, 100, 100)
    COLOR_NEGRO = (0, 0, 0)
    TARGETS = {'OEE': 72.0, 'DISPONIBILIDAD': 90.0, 'PERFORMANCE': 80.0, 'CALIDAD': 99.5}

    def obtener_color(valor, kpi_name):
        meta = TARGETS.get(kpi_name, 0)
        if pd.isna(valor): return COLOR_NEGRO
        if valor >= meta: return (40, 167, 69)
        elif valor >= (meta - 10): return (230, 126, 34)
        else: return (220, 53, 69)

    class ReportePDF(FPDF):
        def print_kpi(self, etiqueta, valor_num, kpi_name, separador=" | "):
            color_dinamico = obtener_color(valor_num, kpi_name)
            valor_txt = f"{valor_num:.1f}%" if not pd.isna(valor_num) else "0.0%"
            self.set_font("Arial", 'B', 10)
            self.set_text_color(*COLOR_NEGRO)
            self.cell(self.get_string_width(etiqueta), 6, txt=etiqueta, ln=0)
            self.set_text_color(*color_dinamico)
            self.cell(self.get_string_width(valor_txt), 6, txt=valor_txt, ln=0)
            self.set_text_color(*COLOR_NEGRO)
            self.cell(self.get_string_width(separador), 6, txt=separador, ln=0)

    pdf = ReportePDF(orientation='P', unit='mm', format='A4')
    pdf.set_auto_page_break(auto=True, margin=15)

    # --- PÁGINA 1: RESUMEN Y GRÁFICO ---
    pdf.add_page()

    # ENCABEZADO
    pdf.set_font("Arial", 'B', 14)
    pdf.set_text_color(*COLOR_TITULO)
    pdf.cell(120, 8, txt="SECCIÓN GRUPO: CELDA RENAULT", ln=0)
    zona_ar = pytz.timezone('America/Argentina/Cordoba')
    fecha_actual = datetime.now(zona_ar).strftime("%d/%m/%Y %H:%M")
    pdf.set_font("Arial", 'I', 9)
    pdf.set_text_color(*COLOR_GRIS)
    pdf.cell(0, 8, txt=f"Generado el: {fecha_actual}", ln=1, align='R')
    pdf.set_draw_color(150, 150, 150)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(4)

    # SECCIÓN 1: RESUMEN OEE
    pdf.set_font("Arial", 'B', 12)
    pdf.set_text_color(*COLOR_TITULO)
    pdf.cell(0, 8, txt=f"1. Resumen OEE del Grupo (Período: {FECHA_INICIO} al {FECHA_FIN})", ln=True)
    pdf.set_draw_color(*COLOR_TITULO)
    pdf.line(10, pdf.get_y(), 200, pdf.get_y())
    pdf.ln(2)

    pdf.set_font("Arial", 'B', 9)
    pdf.set_text_color(*COLOR_GRIS)
    pdf.cell(0, 6, txt="Objetivos (Target) del mes: OEE: 72.0% | Disp: 90.0% | Perf: 80.0% | Cal: 99.5%", ln=True)
    pdf.ln(2)

    if not df_oee.empty:
        pdf.set_font("Arial", 'B', 10)
        pdf.set_text_color(*COLOR_NEGRO)
        pdf.cell(pdf.get_string_width("Total CELDA RENAULT | "), 6, txt="Total CELDA RENAULT | ", ln=0)
        pdf.print_kpi("OEE: ", total_grupo['OEE'], 'OEE')
        pdf.print_kpi("Disp: ", total_grupo['DISPONIBILIDAD'], 'DISPONIBILIDAD')
        pdf.print_kpi("Perf: ", total_grupo['PERFORMANCE'], 'PERFORMANCE')
        pdf.print_kpi("Cal: ", total_grupo['CALIDAD'], 'CALIDAD', separador="")
        pdf.ln(7)

        for index, fila in df_oee.iterrows():
            pdf.cell(5, 5, txt="", ln=0)
            txt_nombre = f"> {fila['MAQUINA']} | "
            pdf.set_font("Arial", 'B', 9.5)
            pdf.set_text_color(*COLOR_NEGRO)
            pdf.cell(pdf.get_string_width(txt_nombre), 5, txt=txt_nombre, ln=0)
            pdf.print_kpi("OEE: ", fila['OEE'], 'OEE')
            pdf.print_kpi("Disp: ", fila['DISPONIBILIDAD'], 'DISPONIBILIDAD')
            pdf.print_kpi("Perf: ", fila['PERFORMANCE'], 'PERFORMANCE')
            pdf.print_kpi("Cal: ", fila['CALIDAD'], 'CALIDAD', separador="")
            pdf.ln(5)

        # SECCIÓN 2: GRÁFICO
        pdf.ln(6)
        pdf.set_font("Arial", 'B', 12)
        pdf.set_text_color(*COLOR_TITULO)
        pdf.cell(0, 8, txt="2. Comparativa de KPIs entre Máquinas", ln=True)
        pdf.set_draw_color(*COLOR_TITULO)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(4)
        pdf.image('grafico_kpis.png', x=10, w=190)

    # SECCIÓN 3: HORARIOS Y TIEMPO DE APERTURA (TABLA AGRUPADA POR TURNO)
    if not df_turnos.empty:
        pdf.ln(6)
        pdf.set_font("Arial", 'B', 12)
        pdf.set_text_color(*COLOR_TITULO)
        pdf.cell(0, 8, txt="3. Horarios y Tiempo de Apertura", ln=True)
        pdf.set_draw_color(*COLOR_TITULO)
        pdf.line(10, pdf.get_y(), 200, pdf.get_y())
        pdf.ln(4)

        # Anchos de columnas (Suma total = 190 mm)
        w_maq = 45
        w_turno = 20
        w_inicio = 30
        w_cierre = 30
        w_apertura = 32
        w_noregis = 33

        # Encabezado Naranja
        pdf.set_fill_color(211, 84, 0)
        pdf.set_text_color(255, 255, 255)
        pdf.set_draw_color(255, 255, 255)
        pdf.set_font("Arial", 'B', 8.5)

        pdf.cell(w_maq, 6, "Maquina", border=1, fill=True, align='C')
        pdf.cell(w_turno, 6, "Turno", border=1, fill=True, align='C')
        pdf.cell(w_inicio, 6, "Hora Inicio", border=1, fill=True, align='C')
        pdf.cell(w_cierre, 6, "Hora Cierre", border=1, fill=True, align='C')
        pdf.cell(w_apertura, 6, "Apertura Neta", border=1, fill=True, align='C')
        pdf.cell(w_noregis, 6, "No Registrado", border=1, fill=True, align='C')
        pdf.ln()

        # Filas de Datos
        pdf.set_text_color(*COLOR_NEGRO)
        pdf.set_font("Arial", '', 8)
        pdf.set_draw_color(210, 210, 210)

        fill_row = False
        turno_actual = None

        for _, fila in df_turnos.iterrows():
            # Si cambia de turno, podemos alternar fondo o mantener zebra striping
            if fill_row:
                pdf.set_fill_color(248, 248, 248)
            else:
                pdf.set_fill_color(255, 255, 255)

            pdf.cell(w_maq, 5.5, str(fila['MAQUINA']), border=1, fill=True, align='L')
            pdf.cell(w_turno, 5.5, str(fila['TURNO']), border=1, fill=True, align='C')
            pdf.cell(w_inicio, 5.5, str(fila['TXT_INICIO']), border=1, fill=True, align='C')
            pdf.cell(w_cierre, 5.5, str(fila['TXT_CIERRE']), border=1, fill=True, align='C')
            pdf.cell(w_apertura, 5.5, str(fila['TXT_APERTURA_NETA']), border=1, fill=True, align='C')
            pdf.cell(w_noregis, 5.5, str(fila['TXT_NO_REGISTRADO']), border=1, fill=True, align='C')
            pdf.ln()
            fill_row = not fill_row

    nombre_pdf = f"Reporte_CELDA_RENAULT_{FECHA_INICIO}_al_{FECHA_FIN}.pdf"
    pdf.output(nombre_pdf)
    print(f"¡PDF Generado exitosamente con orden por Turno! Descárgalo aquí: {nombre_pdf}")

except Exception as e:
    print("❌ Error crítico procesando la base de datos SQL.")
    print(f"Detalle técnico: {e}")